# flowsurface Replay API 疎通確認

「セルを全て実行」してから UI を操作してください。

In [ ]:
%pip install -q ipywidgets httpx

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown
import httpx
import json

In [ ]:
# ── 接続設定 ─────────────────────────────────────────────────────
api_url_w = widgets.Text(
    value="http://127.0.0.1:9876",
    description="API URL:",
    layout=widgets.Layout(width="420px"),
)

conn_label = widgets.HTML(value="<i style='color:gray'>未確認</i>")
check_btn  = widgets.Button(description="✓ 接続確認", button_style="info")

def on_check(b):
    try:
        r  = httpx.get(f"{api_url_w.value}/api/replay/status", timeout=2.0)
        ok = r.status_code == 200
        conn_label.value = (
            f"<b style='color:green'>✅ 接続成功（status={r.status_code}）</b>"
            if ok
            else f"<b style='color:red'>❌ 接続失敗: status={r.status_code}</b>"
        )
    except Exception as e:
        conn_label.value = f"<b style='color:red'>❌ 接続失敗: {e}</b>"

check_btn.on_click(on_check)

# ── エンドポイント一覧 ────────────────────────────────────────────
endpoints_list = [
    # ── Replay 制御 ──────────────────────────────────────────────
    ("GET /api/replay/status",                            "リプレイ状態取得"),
    ("POST /api/replay/toggle",                           "リプレイ開始/停止"),
    ("POST /api/replay/play",                             "期間指定リプレイ開始"),
    ("POST /api/replay/pause",                            "一時停止"),
    ("POST /api/replay/resume",                           "再開"),
    ("POST /api/replay/step-forward",                     "次へ"),
    ("POST /api/replay/step-backward",                    "前へ"),
    ("POST /api/replay/speed",                            "速度変更"),
    # ── 仮想約定エンジン ─────────────────────────────────────────
    ("GET /api/replay/state",                             "仮想取引所の状態取得"),
    ("GET /api/replay/portfolio",                         "ポートフォリオ"),
    ("GET /api/replay/orders",                            "注文一覧（仮想）"),
    ("POST /api/replay/order",                            "注文登録（仮想）"),
    # ── App 制御 ─────────────────────────────────────────────────
    ("POST /api/app/save",                                "状態保存"),
    ("POST /api/app/screenshot",                          "スクリーンショット"),
    # ── 認証 ─────────────────────────────────────────────────────
    ("GET /api/auth/tachibana/status",                    "セッション確認"),
    ("POST /api/auth/tachibana/logout",                   "ログアウト"),
    # ── ペイン CRUD ───────────────────────────────────────────────
    ("GET /api/pane/list",                                "ペイン一覧"),
    ("GET /api/pane/chart-snapshot",                      "チャートスナップショット"),
    ("POST /api/pane/split",                              "ペイン分割"),
    ("POST /api/pane/close",                              "ペイン閉じる"),
    ("POST /api/pane/set-ticker",                         "銘柄変更"),
    ("POST /api/pane/set-timeframe",                      "時間足変更"),
    # ── その他 ───────────────────────────────────────────────────
    ("GET /api/notification/list",                        "通知一覧"),
    ("POST /api/sidebar/select-ticker",                   "サイドバー銘柄選択"),
    ("POST /api/sidebar/open-order-pane",                 "注文ペイン開く"),
    # ── 立花証券 ─────────────────────────────────────────────────
    ("GET /api/buying-power",                             "買付余力取得"),
    ("POST /api/tachibana/order",                         "注文登録（立花）"),
    ("GET /api/tachibana/orders",                         "注文一覧（立花）"),
    ("GET /api/tachibana/order/{id}",                     "注文詳細（立花）"),
    ("POST /api/tachibana/order/correct",                 "注文訂正（立花）"),
    ("POST /api/tachibana/order/cancel",                  "注文キャンセル（立花）"),
    ("GET /api/tachibana/holdings",                       "保有株数"),
    # ── テスト専用（debug ビルドのみ）────────────────────────────
    ("POST /api/test/tachibana/delete-persisted-session", "セッション削除（テスト用）"),
]

endpoint_options = [f"{p} — {d}" for p, d in endpoints_list]
endpoint_labels  = {f"{p} — {d}": (p, d) for p, d in endpoints_list}

endpoint_dd = widgets.Dropdown(
    options=endpoint_options,
    description="エンドポイント:",
    layout=widgets.Layout(width="700px"),
    style={"description_width": "120px"},
)

# ── パラメータフォーム ────────────────────────────────────────────
params_box     = widgets.VBox([])
_param_widgets = {}

def _text(desc, val=""):
    return widgets.Text(
        description=desc, value=val,
        layout=widgets.Layout(width="560px"),
        style={"description_width": "260px"},
    )

def _number(desc, min_v=0.001, max_v=1000.0, step=0.001, val=0.01):
    return widgets.FloatText(
        description=desc, value=val,
        layout=widgets.Layout(width="420px"),
        style={"description_width": "200px"},
    )

def _dd(desc, opts, val=None):
    return widgets.Dropdown(
        description=desc, options=opts, value=val or opts[0],
        layout=widgets.Layout(width="420px"),
        style={"description_width": "200px"},
    )

def _build_params(path_only, method):
    p = {}
    if method == "GET":
        if "chart-snapshot" in path_only:
            p["pane_id"] = _text("pane_id", "00000000-0000-0000-0000-000000000001")
        elif "/api/tachibana/order/{id}" in path_only:
            p["order_num"] = _text("order_num")
            p["eig_day"]   = _text("eig_day (YYYYMMDD, 省略可)")
        elif "tachibana/orders" in path_only:
            p["eig_day"] = _text("eig_day (YYYYMMDD, 省略可)")
        elif "tachibana/holdings" in path_only:
            p["issue_code"] = _text("issue_code", "7203")
    elif method == "POST":
        if "replay/play" in path_only:
            p["start"] = _text("start (YYYY-MM-DD HH:MM:SS)", "2024-01-01 09:00:00")
            p["end"]   = _text("end   (YYYY-MM-DD HH:MM:SS)", "2024-01-01 15:30:00")
        elif "replay/order" in path_only:
            p["ticker"] = _text("ticker", "BinanceLinear:BTCUSDT")
            p["side"]   = _dd("side", ["buy", "sell"])
            p["qty"]    = _number("qty", 0.001, 1000.0, 0.001, 0.01)
        elif "replay/speed" in path_only:
            p["speed"] = _number("speed", 0.1, 100.0, 0.5, 1.0)
        elif "pane/split" in path_only:
            p["pane_id"] = _text("pane_id", "00000000-0000-0000-0000-000000000001")
            p["axis"]    = _dd("axis", ["Vertical", "Horizontal"])
        elif "pane/close" in path_only:
            p["pane_id"] = _text("pane_id", "00000000-0000-0000-0000-000000000001")
        elif "pane/set-ticker" in path_only:
            p["pane_id"] = _text("pane_id", "00000000-0000-0000-0000-000000000001")
            p["ticker"]  = _text("ticker", "BinanceLinear:BTCUSDT")
        elif "pane/set-timeframe" in path_only:
            p["pane_id"]   = _text("pane_id", "00000000-0000-0000-0000-000000000001")
            p["timeframe"] = _text("timeframe", "1m")
        elif "sidebar/select-ticker" in path_only:
            p["pane_id"] = _text("pane_id", "00000000-0000-0000-0000-000000000001")
            p["ticker"]  = _text("ticker", "BinanceLinear:BTCUSDT")
            p["kind"]    = _text("kind (省略可)")
        elif "sidebar/open-order-pane" in path_only:
            p["kind"] = _dd("kind", ["OrderEntry", "OrderList", "BuyingPower"])
        elif path_only == "/api/tachibana/order" or (
            "tachibana/order" in path_only
            and "correct" not in path_only
            and "cancel" not in path_only
        ):
            p["issue_code"]      = _text("issue_code", "7203")
            p["qty"]             = _text("qty", "100")
            p["side"]            = _dd("side", ["buy", "sell"])
            p["price"]           = _text("price (0=成行)", "0")
            p["account_type"]    = _text("account_type (省略可, 既定=1)")
            p["market_code"]     = _text("market_code (省略可, 既定=00)")
            p["condition"]       = _text("condition (省略可, 既定=0)")
            p["cash_margin"]     = _text("cash_margin (省略可, 既定=0)")
            p["expire_day"]      = _text("expire_day (省略可, 既定=0)")
            p["second_password"] = _text("second_password")
        elif "tachibana/order/correct" in path_only:
            p["order_number"]    = _text("order_number")
            p["eig_day"]         = _text("eig_day (YYYYMMDD)")
            p["condition"]       = _text("condition (省略可, 既定=*)")
            p["price"]           = _text("price (省略可, 既定=*)")
            p["qty"]             = _text("qty (省略可, 既定=*)")
            p["expire_day"]      = _text("expire_day (省略可, 既定=*)")
            p["second_password"] = _text("second_password")
        elif "tachibana/order/cancel" in path_only:
            p["order_number"]    = _text("order_number")
            p["eig_day"]         = _text("eig_day (YYYYMMDD)")
            p["second_password"] = _text("second_password")
    return p

def on_endpoint_change(change):
    global _param_widgets
    if endpoint_dd.value is None:
        _param_widgets = {}
        params_box.children = []
        return
    path, _ = endpoint_labels[endpoint_dd.value]
    method    = "GET" if path.startswith("GET") else "POST"
    path_only = path.split(" ", 1)[1]
    _param_widgets      = _build_params(path_only, method)
    params_box.children = list(_param_widgets.values())

endpoint_dd.observe(on_endpoint_change, names="value")
on_endpoint_change(None)

# ── レスポンス出力 ────────────────────────────────────────────────
resp_out = widgets.Output()
send_btn = widgets.Button(description="📤 リクエスト送信", button_style="primary")

def on_send(b):
    resp_out.clear_output(wait=True)
    if endpoint_dd.value is None:
        return
    path, _   = endpoint_labels[endpoint_dd.value]
    method     = "GET" if path.startswith("GET") else "POST"
    path_only  = path.split(" ", 1)[1]
    params     = {k: w.value for k, w in _param_widgets.items()}

    with resp_out:
        try:
            if method == "GET" and "chart-snapshot" in path_only:
                url  = f"{api_url_w.value}/api/pane/chart-snapshot?pane_id={params.get('pane_id', '')}"
                resp = httpx.get(url, timeout=5.0)
            elif method == "GET" and "/api/tachibana/order/{id}" in path_only:
                url  = f"{api_url_w.value}/api/tachibana/order/{params.get('order_num', '')}"
                day  = params.get("eig_day", "")
                if day:
                    url += f"?eig_day={day}"
                resp = httpx.get(url, timeout=5.0)
            elif method == "GET" and "tachibana/orders" in path_only:
                url  = f"{api_url_w.value}/api/tachibana/orders"
                day  = params.get("eig_day", "")
                if day:
                    url += f"?eig_day={day}"
                resp = httpx.get(url, timeout=5.0)
            elif method == "GET" and "tachibana/holdings" in path_only:
                url  = f"{api_url_w.value}/api/tachibana/holdings?issue_code={params.get('issue_code', '')}"
                resp = httpx.get(url, timeout=5.0)
            elif method == "GET":
                resp = httpx.get(f"{api_url_w.value}{path_only}", timeout=5.0)
            else:
                body = {k: v for k, v in params.items() if v != ""}
                resp = httpx.post(f"{api_url_w.value}{path_only}", json=body, timeout=5.0)

            code = resp.status_code
            icon = "✅" if 200 <= code < 300 else "⚠️"
            try:
                text = json.dumps(resp.json(), indent=2, ensure_ascii=False)
            except Exception:
                text = resp.text
            display(Markdown(f"### レスポンス {icon}\n\n**Status**: `{code}`\n\n```json\n{text}\n```"))
        except Exception as e:
            display(Markdown(f"❌ エラー: `{e}`"))

send_btn.on_click(on_send)

# ── 全体レイアウト ────────────────────────────────────────────────
display(widgets.VBox([
    widgets.HTML("<h3>接続設定</h3>"),
    api_url_w,
    widgets.HBox([check_btn, conn_label], layout=widgets.Layout(align_items="center", gap="12px")),
    widgets.HTML("<hr><h3>エンドポイント選択</h3>"),
    endpoint_dd,
    widgets.HTML("<h4>パラメータ</h4>"),
    params_box,
    widgets.HTML("<hr>"),
    send_btn,
    resp_out,
]))